# Agent

Run `pipeline.rag_agent.answer_question` for one question and one source file.
Replace the question template before running. Set `SHOW_TEXT = True` to print the answer and the citation.
The last cell deletes this collection.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

from helpers import accepted_pdfs, new_session_id, preview, timed

from pipeline.embedder import embed_document
from pipeline.extractor import extract

SHOW_TEXT = False
pdf_dir = (
    Path(os.environ["NOTEBOOK_PDF_DIR"])
    if os.environ.get("NOTEBOOK_PDF_DIR")
    else ROOT / "notebooks" / "inputs"
)
session_id = new_session_id()
documents = []

with timed("setup") as setup:
    accepted, rejected = accepted_pdfs(pdf_dir)
    for path in accepted:
        try:
            document = await extract(path)
        except Exception as exc:
            print(path.name, "failure", type(exc).__name__, exc)
            continue
        if document.metadata.total_chunks == 0:
            print(path.name, "failure", "RuntimeError", "produced no text")
            continue
        await embed_document(document, session_id)
        documents.append(document)

print(session_id)
print(f"setup: {setup['seconds']:.3f}s")
print(f"ready: {len(documents)}  rejected: {len(rejected)}")

In [ ]:
from models.query import QuestionType, UserQuestion

source = documents[0].metadata.filename if documents else ""
question = UserQuestion(
    label="Field",
    question="Copy the requested value from the document.",
    question_type=QuestionType.EXTRACTION,
)
print(source)
print(question.label, question.question_type.value)

In [ ]:
from pipeline.rag_agent import answer_question

with timed("answer") as answer_time:
    result = answer_question(question, session_id, source) if source else None

print(f"answer: {answer_time['seconds']:.3f}s")

In [ ]:
if result is not None:
    citation = result.citation
    source_name = result.answer_source.value if result.answer_source else None
    print("label", result.question.label)
    print("question_type", result.question.question_type.value)
    print("confidence", result.confidence.value)
    print("answer_source", source_name)
    print("citation_page", citation.page if citation else None)
    print("citation_score", citation.score if citation else None)
    print("citation_chars", len(citation.text) if citation else 0)
    if SHOW_TEXT:
        print(preview(result.answer, SHOW_TEXT))
        print(preview(citation.text if citation else None, SHOW_TEXT))

In [ ]:
from pipeline.embedder import delete_collection

delete_collection(session_id)
print(session_id, "deleted")